In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
# ------------------------------------------------------------
# Parámetros del sistema (modificables)
# ------------------------------------------------------------
T = 10.0                     # Tiempo límite para nuevas llegadas
simulation_time = 50.0       # Tiempo máximo de simulación (seguridad)
n_replicas = 1000            # Número de réplicas para estimar promedio e IC

# ------------------------------------------------------------
# Funciones de llegadas (Poisson no homogéneo)
# ------------------------------------------------------------
def lambda_t(t):
    """
    Intensidad de llegadas en función del tiempo.
    Ejemplo: λ(t) = 1 + 0.5*sin(t)  (siempre positiva).
    """
    return 1.0 + 0.5 * np.sin(t)

def lambda_max():
    """
    Valor máximo de λ(t) en el rango de simulación (para el método de thinning).
    Como λ(t) ∈ [0.5, 1.5] (aprox.), usamos 1.5.
    """
    return 1.5

In [4]:
def next_arrival_thinning(t_current):
    """
    Genera el próximo tiempo de llegada dado el tiempo actual t_current,
    para un proceso de Poisson no homogéneo con intensidad λ(t).
    Utiliza el método de thinning (aceptación-rechazo) con tasa máxima λ_max.
    
    Pasos:
    1. Generar un tiempo de llegada candidato desde un Poisson homogéneo
       con tasa λ_max, sumando un interarrival exponencial.
    2. Aceptar el candidato con probabilidad λ(t_candidate) / λ_max.
    3. Repetir hasta aceptar.
    """
    t = t_current
    while True:
        # Interarrival exponencial con tasa λ_max
        t += np.random.exponential(1.0 / lambda_max())
        # Probabilidad de aceptación
        if np.random.rand() < lambda_t(t) / lambda_max():
            return t

In [5]:
# ------------------------------------------------------------
# Distribución del tiempo de servicio G (exponencial media=1)
# ------------------------------------------------------------
def service_time():
    """Genera un tiempo de servicio ~ Exp(1) (media = 1)."""
    return np.random.exponential(1.0)

In [6]:
# ------------------------------------------------------------
# Simulación de una réplica del sistema
# ------------------------------------------------------------
def simular_una_replica():
    """
    Realiza una simulación completa del sistema desde t=0 hasta que
    no queden clientes en el sistema y no haya más llegadas.
    Retorna:
        - sojourn_times: lista de tiempos de estancia de cada cliente.
        - Tp: tiempo extra después de T que termina el último cliente.
    """
    # Inicialización de variables (según el algoritmo)
    t = 0.0                # tiempo actual
    NA = 0                 # número de llegadas hasta ahora
    ND = 0                 # número de salidas hasta ahora
    n = 0                  # número de clientes en el sistema (cola + servicio)
    
    # Generar primera llegada
    tA = next_arrival_thinning(0.0)
    tD = np.inf            # inicialmente no hay cliente en servicio
    
    # Diccionarios para almacenar tiempos de llegada y salida
    # (usamos listas, pero los diccionarios serían útiles si se requiere
    # acceso aleatorio por índice de cliente; aquí basta con listas ordenadas)
    arrival_times = []     # lista de tiempos de llegada (índice = NA-1)
    departure_times = []   # lista de tiempos de salida (índice = ND-1)
    
    # Simulación por eventos
    while True:
        # Caso 1: la próxima llegada ocurre antes que la próxima salida
        #          y además la llegada es antes o en T
        if tA <= tD and tA <= T:
            # Actualizar tiempo actual
            t = tA
            # Incrementar contador de llegadas
            NA += 1
            # Aumentar el estado del sistema
            n += 1
            # Registrar tiempo de llegada del nuevo cliente
            arrival_times.append(t)
            
            # Generar la siguiente llegada
            tA = next_arrival_thinning(t)
            
            # Si el sistema estaba vacío (n == 1 después de esta llegada)
            # entonces este cliente entra directamente al servicio
            if n == 1:
                serv = service_time()
                tD = t + serv
        
        # Caso 2: la próxima salida ocurre antes que la próxima llegada
        #          y la salida es antes o en T
        elif tD < tA and tD <= T:
            t = tD
            n -= 1
            ND += 1
            # Registrar tiempo de salida del cliente que termina
            # (el cliente que sale es el que llegó más temprano entre los presentes)
            departure_times.append(t)
            
            # Si después de la salida quedan clientes en cola
            if n > 0:
                # El siguiente cliente (el que estaba primero en cola) comienza servicio
                serv = service_time()
                tD = t + serv
            else:
                tD = np.inf
        
        # Caso 3: ya pasó T, no hay más llegadas, pero aún hay clientes en el sistema
        elif min(tA, tD) > T and n > 0:
            # Solo pueden ocurrir salidas
            t = tD
            n -= 1
            ND += 1
            departure_times.append(t)
            
            if n > 0:
                serv = service_time()
                tD = t + serv
            else:
                tD = np.inf
        
        # Caso 4: ya pasó T, sistema vacío (n=0) → fin de la simulación
        elif min(tA, tD) > T and n == 0:
            # Calcular Tp = tiempo extra después de T que termina el último cliente
            # El último tiempo de salida registrado (si existe) o el tiempo actual
            if departure_times:
                last_departure = max(departure_times)
            else:
                last_departure = t
            Tp = max(last_departure - T, 0.0)
            break
        
        # Seguridad: si el tiempo supera un límite, salir
        if t > simulation_time:
            print("Advertencia: se alcanzó el tiempo máximo de simulación")
            Tp = 0.0
            break
    
    # Calcular tiempos de estancia (sojourn times) para cada cliente
    # Nota: el número de llegadas y salidas debe ser igual
    sojourn_times = [departure_times[i] - arrival_times[i] for i in range(len(arrival_times))]
    
    return sojourn_times, Tp

In [7]:
# ------------------------------------------------------------
# Realizar n_replicas simulaciones independientes
# ------------------------------------------------------------
all_sojourn = []      # lista de listas (cada réplica tiene sus tiempos)
all_Tp = []           # lista de tiempos extra por réplica

for rep in range(n_replicas):
    soj, tp = simular_una_replica()
    all_sojourn.extend(soj)   # acumulamos todos los tiempos de todas las réplicas
    all_Tp.append(tp)

# Promedios globales
avg_sojourn = np.mean(all_sojourn)
avg_Tp = np.mean(all_Tp)

# Intervalos de confianza al 95% (usando percentiles o normal)
ic_sojourn = np.percentile(all_sojourn, [2.5, 97.5])
ic_Tp = np.percentile(all_Tp, [2.5, 97.5])

print("="*50)
print(f"Número de réplicas: {n_replicas}")
print(f"Tiempo promedio en el sistema: {avg_sojourn:.4f}")
print(f"IC 95% para tiempo en sistema: ({ic_sojourn[0]:.4f}, {ic_sojourn[1]:.4f})")
print(f"Tiempo extra promedio después de T: {avg_Tp:.4f}")
print(f"IC 95% para tiempo extra: ({ic_Tp[0]:.4f}, {ic_Tp[1]:.4f})")

Número de réplicas: 1000
Tiempo promedio en el sistema: 3.2012
IC 95% para tiempo en sistema: (0.0956, 11.0154)
Tiempo extra promedio después de T: 3.5860
IC 95% para tiempo extra: (0.0000, 12.4996)
